# Self-supervised learning + MLP with PyTorch

Denoising pretraining learns spectral representations without age labels. A frozen encoder feeds a target-city MLP. This is a fixed configuration experiment on **five previously used spatial development folds × three label samples**, with **25 Amsterdam labels per class**. These are development results, not independent-test estimates.

The older scikit-learn experiment is preserved in `self_supervised_sklearn.ipynb`; its results remain under `data/self_supervised/`. The PyTorch results use `data/self_supervised_torch/`.


In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay
from scripts import self_supervised_torch as experiment
from pathlib import Path
import sys
import inspect
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
OUT = experiment.OUT
print('PyTorch:', experiment.torch.__version__)
display(pd.Series(experiment.CONFIG, name='fixed configuration'))

## Protocol and leakage controls

Use the baseline's explicit 54-feature allowlist. Coordinates establish spatial groups but are not predictors. Construction year, class and label-derived fields never enter self-supervision. Pretrain on up to 20,000 Madrid and 20,000 eligible Amsterdam locations per fold; exclude every held-out Amsterdam block and location. Madrid labels are never read.

A group-disjoint 10% split **inside the pretraining pool** selects the reconstruction checkpoint. Median imputation and standardization fit only on its fit portion. The denoising autoencoder is 54 → 64 → 16 → 64 → 54 with ReLU hidden layers, 20% fresh input masking and full clean-input MSE reconstruction. AdamW trains for at most 60 epochs with patience eight. A fixed checkpoint mask makes loss comparisons consistent.

Freeze the 16-dimensional encoder. A 16 → 32 → 4 classifier uses ReLU, 20% dropout, AdamW and a fixed 300 full-batch epochs. Balanced 25/class samples need no class weights. Raw logits enter cross entropy with class indices 0–3. No outer labels choose epochs, architecture or regularization. Eval mode and no-grad disable dropout and gradient recording at inference. Seeds and deterministic CPU operations are set; exact reproducibility across releases or platforms is not promised.

The raw control uses a 54 → 32 → 4 PyTorch MLP with the same optimizer, dropout and training budget, and preprocessing fitted on its 100 labels. This compares complete procedures, including SSL access to unlabeled data, rather than isolating the pretraining objective. The RF blend additionally uses Madrid labels and therefore has a different supervision budget.


## Implementation

The executable implementation is in `scripts/self_supervised_torch.py`. The following cell displays the actual definitions used by this notebook, so the model and training loops are inspectable without a second implementation drifting out of sync.


In [ ]:
for obj in [experiment.seed_all, experiment.DenoisingEncoder, experiment.TorchHead, experiment.head]:
    display(Markdown('```python\n' + inspect.getsource(obj) + '\n```'))

## Run or load the reproducible experiment

Set `RETRAIN = True` to rerun all five folds. Default loading verifies the script and input hashes before trusting saved artifacts. A missing or stale run is retrained. A full run writes memberships, probabilities, per-class metrics, loss histories, split audits, example weights and a hash manifest.


In [ ]:
RETRAIN = False
manifest_path = OUT / 'experiment.json'
fresh = False
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    fresh = ('outputs' in manifest
             and manifest['code_sha256'] == experiment.sha(experiment.__file__)
             and all(experiment.sha(ROOT / p) == h for p, h in manifest['inputs'].items()))
if RETRAIN or not fresh:
    experiment.run()
manifest = json.loads(manifest_path.read_text())
for name, expected in manifest['outputs'].items():
    assert experiment.sha(OUT / name) == expected, name
print(
    f"Verified run: {manifest['elapsed_seconds']:.1f} seconds; independent_test={manifest['independent_test']}")

## Scores and prediction verification

Average the three repetitions within each fold first, then give the five folds equal weight. Fold standard deviation is descriptive, not a confidence interval: folds overlap in training data and repetitions share validation locations.


In [ ]:
runs = pd.read_csv(OUT / 'run_metrics.csv')
folds = pd.read_csv(OUT / 'fold_metrics.csv')
summary = pd.read_csv(OUT / 'summary.csv', index_col=0)
predictions = pd.read_parquet(OUT / 'predictions.parquet')
assert len(runs) == 45
for (method, fold, repetition), frame in predictions.groupby(['method', 'fold', 'repetition']):
    p = frame[[f'p{c}' for c in experiment.CLASSES]].to_numpy()
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1), 1)
    np.testing.assert_array_equal(
        frame.y_pred, experiment.CLASSES[p.argmax(axis=1)])
    recomputed = f1_score(frame.y_true, frame.y_pred,
                          labels=experiment.CLASSES, average='macro')
    saved = runs.query(
        'method == @method and fold == @fold and repetition == @repetition').macro_f1.item()
    assert np.isclose(recomputed, saved)
recomputed_summary = runs.groupby(
    ['method', 'fold']).macro_f1.mean().groupby('method').mean()
np.testing.assert_allclose(
    summary.loc[recomputed_summary.index, 'macro_f1'], recomputed_summary)
display(summary.round(4))
display(pd.read_csv(OUT / 'paired_fold_scores.csv').round(4))
audit = pd.read_csv(OUT / 'split_audit.csv')
assert audit[['overlap_locations', 'overlap_spatial_groups']].eq(0).all().all()
assert audit.checkpoint_mse.lt(audit.checkpoint_zero_mse).all()
display(audit)

In [ ]:
history = pd.read_csv(OUT / 'pretraining_history.csv')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for fold, frame in history.groupby('fold'):
    axes[0].plot(frame.epoch, frame.checkpoint_mse, label=f'Fold {fold+1}')
axes[0].set(xlabel='Pretraining epoch', ylabel='Checkpoint reconstruction MSE',
            title='Training-domain reconstruction')
axes[0].legend()
for method, frame in folds.groupby('method'):
    axes[1].plot(frame.fold+1, frame.macro_f1, marker='o', label=method)
axes[1].set(xlabel='Spatial fold', ylabel='Macro F1',
            title='Matched development comparison')
axes[1].legend()
plt.tight_layout()
plt.show()

## Class-level results

Per-class metrics below average within folds and then across folds. The normalized confusion matrix uses repetition zero to avoid counting each validation location three times; pooled confusion rates and equal-fold average metrics have different weighting.


In [ ]:
class_metrics = pd.read_csv(OUT / 'class_metrics.csv')
class_average = class_metrics.groupby(['method', 'fold', 'age_class'])[
    ['precision', 'recall', 'f1']].mean().groupby(['method', 'age_class']).mean()
display(class_average.round(4))
frame = predictions.query("method == 'ssl_mlp' and repetition == 0")
ConfusionMatrixDisplay.from_predictions(
    frame.y_true, frame.y_pred, labels=experiment.CLASSES, normalize='true', cmap='Blues', values_format='.2f')
plt.title('SSL + MLP: held-out predictions (one repetition)')
plt.tight_layout()
plt.show()

## Conclusion


In [ ]:
ssl = summary.loc['ssl_mlp']
raw = summary.loc['raw_mlp']
rf = summary.loc['original_blend']
display(Markdown(
    f"**PyTorch SSL + MLP: {ssl.macro_f1:.4f} macro F1 ± {ssl.macro_f1_fold_sd:.4f} fold SD**, "
    f"weighted F1 {ssl.weighted_f1:.4f}, accuracy {ssl.accuracy:.4f}. "
    f"The paired mean differences are {ssl.macro_f1-raw.macro_f1:+.4f} versus the raw PyTorch MLP "
    f"and {ssl.macro_f1-rf.macro_f1:+.4f} versus the RF blend. "
    + ('This configuration does not support replacing the recommended RF blend. ' if ssl.macro_f1 < rf.macro_f1
       else 'This development gain requires confirmation on untouched geographic data. ')
    + 'Reconstruction improvement establishes that the pretext task was learned; it does not establish age-discriminative representations. '
      'A frozen bottleneck may discard useful age information, but these results do not identify the cause. '
      'One fixed architecture and budget cannot rule out fine-tuning or other self-supervised objectives. '
      'Previously used spatial folds, touching blocks and no untouched competition test limit generalization claims.'))

## Guidance and references

Applied the installed **scikit-learn** skill (`.agents/skills/scikit-learn/SKILL.md` and its model-evaluation reference) for train-only preprocessing, spatial independence and reporting; the **Firecrawl developer-index** skill supported current documentation lookup. PyTorch API contracts were checked against official [AdamW](https://docs.pytorch.org/docs/2.15/generated/torch.optim.AdamW.html), [CrossEntropyLoss](https://docs.pytorch.org/docs/2.15/generated/torch.nn.CrossEntropyLoss.html) and [reproducibility](https://docs.pytorch.org/docs/2.15/notes/randomness.html) documentation. The project lock records the installed CPU PyTorch build. This notebook uses a denoising pretext task; no claim of a novel SSL method is made.
